# CSV / TSV の文字化けと列ずれを避ける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/A_colab_python_tips/csv_io.ipynb)

NLP クックブック A-13 ／ Colab と Python の小技

日本語のテキストを CSV で保存して Excel で開くと文字化けする。テキスト中の改行やカンマで列がずれる。

In [1]:
import csv
import pandas as pd

df = pd.DataFrame({"id": [1, 2, 3], "text": ["普通の文です。", "カンマ, と \"引用符\" を含む", "改行を\n含む文"], "label": ["A", "B", "A"]})

df.to_csv("data.csv", index=False)                                # 既定は UTF-8 (BOM なし)。pandas 同士なら問題ない
df.to_csv("for_excel.csv", index=False, encoding="utf-8-sig")     # Excel で開く人に渡すなら BOM 付き UTF-8
print(open("data.csv", encoding="utf-8").read())
print(pd.read_csv("data.csv").equals(df))

id,text,label
1,普通の文です。,A
2,"カンマ, と ""引用符"" を含む",B
3,"改行を
含む文",A

True


In [2]:
# TSV: テキスト中のタブと改行に注意。引用符の処理を無効にするなら、事前に置換しておく
safe = df.assign(text=df["text"].str.replace(r"[\t\n\r]+", " ", regex=True))
safe.to_csv("data.tsv", sep="\t", index=False, quoting=csv.QUOTE_NONE)
back = pd.read_csv("data.tsv", sep="\t", quoting=csv.QUOTE_NONE)
print(back)

# 読み込み時の定番の落とし穴: 文字列の列が勝手に数値や NaN になる
tricky = pd.DataFrame({"code": ["001", "002", "NA", "null"], "word": ["なし", "None", "nan", ""]})
tricky.to_csv("tricky.csv", index=False)
print(pd.read_csv("tricky.csv").to_dict("list"))
print(pd.read_csv("tricky.csv", dtype=str, keep_default_na=False).to_dict("list"))

   id              text label
0   1           普通の文です。     A
1   2  カンマ, と "引用符" を含む     B
2   3           改行を 含む文     A
{'code': [1.0, 2.0, nan, nan], 'word': ['なし', nan, nan, nan]}
{'code': ['001', '002', 'NA', 'null'], 'word': ['なし', 'None', 'nan', '']}


- Windows の Excel が出力した CSV は Shift_JIS (`encoding="cp932"`) のことが多い。
- `dtype=str, keep_default_na=False` を付けると、「001」が 1 になったり、「NA」「null」「None」という文字列が欠損値に化けたりするのを防げる。
- 対訳データなどで 1 行 1 文のプレーンテキストを使う場合も、文中の改行は必ず除いておく（行数がずれると対応が全部壊れる）。